In [5]:
print("FOIL - 4CB23AI071")

import wittgenstein as lw  
import pandas as pd
import math 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score   

data = pd.read_csv("play_tennis.csv")

data['play'] = data['play'].map({'Yes': True, 'No': False})

train, test = train_test_split(data, test_size=0.2, random_state=42)

def foil_gain(p0, n0, p1, n1):
    if p1 == 0 or p1 + n1 == 0:
        return -1
    old = p0 / (p0 + n0)
    new = p1 / (p1 + n1)
    
    if old == 0:
        return -1
        
    return p1 * math.log2(new / old)

def foil(train):
    target = "play"
    attributes = [c for c in train.columns if c != target]
    positives = train[train[target] == True].copy()
    negatives = train[train[target] == False].copy()
    rules = []
    
    while len(positives) > 0:
        examples = pd.concat([positives, negatives])
        rule = []
        
        while True:
            if len(examples[examples[target] == False]) == 0:
                break
            
            best_gain = -1
            best_condition = None
            best_data = None
            p0 = len(examples[examples[target] == True])
            n0 = len(examples[examples[target] == False])

            for col in attributes:
                if col in [x[0] for x in rule]:
                    continue
                for value in examples[col].unique():
                    data_subset = examples[examples[col] == value]
                    p1 = len(data_subset[data_subset[target] == True])
                    n1 = len(data_subset[data_subset[target] == False])
                    gain = foil_gain(p0, n0, p1, n1)
                    
                    if gain > best_gain:
                        best_gain = gain
                        best_condition = (col, value)
                        best_data = data_subset
                        
            if best_condition is None or best_gain <= 0:
                break
            
            rule.append(best_condition)
            examples = best_data
            
        if rule:
            rules.append(rule)
            covered = examples[examples[target] == True]
            positives = positives[~positives.index.isin(covered.index)]
        else: 
            break
            
    return rules

rules = foil(train)

print("==== FOIL Learned Rules ====")
for i, rule in enumerate(rules, 1):
    conditions = " AND ".join(f"{col}={value}" for col, value in rule)
    print(f"Rule {i}: IF {conditions} THEN PlayTennis=Yes")

def predict(row, rules):
    for rule in rules:
        if all(row[col] == value for col, value in rule):
            return True
    return False

predictions = [
    predict(row, rules)
    for _, row in test.iterrows()
]

print("\nActual    : ", list(test["play"]))
print("Predicted : ", predictions)

accuracy = accuracy_score(test["play"], predictions)
print(f"\nFOIL Accuracy: {accuracy:.2f}")

FOIL - 4CB23AI071
==== FOIL Learned Rules ====
Rule 1: IF outlook=Overcast THEN PlayTennis=Yes
Rule 2: IF humidity=Normal AND outlook=Sunny THEN PlayTennis=Yes
Rule 3: IF wind=Weak AND outlook=Rain THEN PlayTennis=Yes

Actual    :  [True, True, False]
Predicted :  [True, True, False]

FOIL Accuracy: 1.00


In [4]:
pip install wittgenstein pandas scikit-learn

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wittgenstein: filename=wittgenstein-0.3.5-py3-none-any.whl size=99134 sha256=b52b50402acdbef0946f13f58fe6c33754e03f79d056d0eb7d4f09d9b37bf49e
  Stored in directory: c:\users\home\appdata\local\pip\cache\wheels\ed\c4\1c\36ab0b352b6fc249d4bb154081713e3a42c868994e8aaa74de
Successfully built wittgenstein
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
